# M6 · BM25 기준 검색 결과

고정한 다자료 21문항으로 BM25 검색을 확인한다. 전체 자료 검색과 질문의 자료 유형을 미리 적용한 검색을 나눠 본다. 이 노트북은 `data/personal/evaluation/questions_multisource.csv`를 그대로 사용한다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None
sys.path.insert(0, str(ROOT))
from src.bm25_eval import evaluate_bm25

details, summary = evaluate_bm25(ROOT)
summary.sort_values(['scope', 'source_type'])

## 평가 기준

- Okapi BM25를 청크 단위 문자 n-gram(2~4글자)에 적용했다. 한국어 형태소 분석기를 추가 설치하지 않고 조사·띄어쓰기 차이를 어느 정도 견디는 문자 단위를 썼다.
- 한 문서에 청크가 여러 개면 가장 높은 청크 점수를 문서 점수로 썼고, 상위 5개 서로 다른 `doc_id`로 평가했다.
- `source_filtered`는 평가 질문의 자료 유형을 미리 알고 필터링한 결과다. 자료 유형을 모르는 실제 질문과는 조건이 다르므로 `all_sources`도 같이 본다.
- 정답 ID·문항은 고정했고, 임베딩 및 LLM API는 호출하지 않았다.

In [ ]:
display(summary.sort_values(['scope', 'source_type']).round(3))

In [ ]:
plot_data = summary[summary['source_type'] != 'ALL'].copy()
order = ['kosha_guide', 'sif', 'moel_report']
labels = {'kosha_guide': 'KOSHA GUIDE', 'sif': 'SIF', 'moel_report': 'MOEL reports'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, metric, title in zip(axes, ['hit1', 'hit5'], ['Hit@1', 'Hit@5']):
    pivot = plot_data.pivot(index='source_type', columns='scope', values=metric).reindex(order)
    pivot = pivot.rename(index=labels, columns={'all_sources': '전체 자료', 'source_filtered': '자료 유형 필터'})
    pivot.plot(kind='bar', ax=ax, color=['#5276A7', '#43A08C'])
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('정답 문서가 포함된 비율')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='검색 범위', frameon=False)
fig.suptitle('BM25 검색 결과: 자료 유형 필터 유무', y=1.02)
fig.tight_layout()
fig.savefig(ROOT / 'data/personal/figures/bm25_hit_rate_by_source.png', dpi=160, bbox_inches='tight')
plt.show()

## 전체 자료 검색에서 상위 5개에 정답이 없던 질문

In [ ]:
misses = details[(details['scope'] == 'all_sources') & (details['hit5'] == 0)]
display(misses[['qid', 'query', 'gold_doc_ids', 'retrieved_doc_ids']])

## BM25 기준 검색 결과

- 21개 고정 질문의 전체 자료 검색에서 BM25는 Hit@1 13/21(0.619), Hit@5 16/21(0.762), Precision@5 0.152, MRR@5 0.656이었다.
- 질문별 자료 유형을 미리 필터링하면 Hit@5 21/21, MRR@5 0.857로 올라갔다. 이는 자료 유형을 아는 조건이 전체 검색보다 쉽다는 점을 보여준다. 실제 앱에서 유형을 모르는 질문의 점수로 바꿔 말하지 않는다.
- BM25는 단어·문자 조각의 일치를 이용한다. 다른 표현·오탈자·띄어쓰기에서 놓치거나, 흔한 단어가 겹쳐 잘못된 사례를 올릴 수 있다.
- 이 결과는 같은 평가 세트에서 pgvector 결과와 나란히 볼 기준선이다. 검색 점수만으로 최종 RAG 답변의 정확도를 보장하지 않는다.
